In [ ]:
!pip install pycountry
!pip install dash

import pandas as pd
import pycountry
from dash import Dash, html, dcc, callback, Output, Input
import plotly.express as px

In [ ]:
# from google.colab import drive

# drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# streaming_df = pd.read_csv("/content/drive/MyDrive/streaming.csv")
# happiness_df = pd.read_csv("/content/drive/MyDrive/2019.csv")

streaming_df = pd.read_csv("streaming.csv")
happiness_df = pd.read_csv("2019.csv")

In [ ]:
streaming_df.head()

,title,type,genres,releaseYear,imdbId,imdbAverageRating,imdbNumVotes,availableCountries,service
0,The Fifth Element,movie,"Action, Adventure, Sci-Fi",1997.0,tt0119116,7.6,517631.0,"AT, CH, DE",Netflix
1,Kill Bill: Vol. 1,movie,"Action, Crime, Thriller",2003.0,tt0266697,8.2,1223205.0,"AE, AL, AO, AT, AU, AZ, BG, BH, BY, CA, CI, CM...",Netflix
2,Jarhead,movie,"Biography, Drama, War",2005.0,tt0418763,7.0,211839.0,"AD, AE, AG, AL, AO, AT, AZ, BA, BG, BH, BM, BR...",Netflix
3,Unforgiven,movie,"Drama, Western",1992.0,tt0105695,8.2,444402.0,"AU, BA, BG, CZ, HR, HU, MD, ME, MK, NZ, PL, RO...",Netflix
4,Eternal Sunshine of the Spotless Mind,movie,"Drama, Romance, Sci-Fi",2004.0,tt0338013,8.3,1106220.0,"AD, AE, AG, AL, AO, AR, AU, AZ, BA, BB, BE, BG...",Netflix


In [ ]:
happiness_df.head()

,Overall rank,Country or region,Score,GDP per capita,Social support,Healthy life expectancy,Freedom to make life choices,Generosity,Perceptions of corruption
0,1,Finland,7.769,1.340,1.587,0.986,0.596,0.153,0.393
1,2,Denmark,7.600,1.383,1.573,0.996,0.592,0.252,0.410
2,3,Norway,7.554,1.488,1.582,1.028,0.603,0.271,0.341
3,4,Iceland,7.494,1.380,1.624,1.026,0.591,0.354,0.118
4,5,Netherlands,7.488,1.396,1.522,0.999,0.557,0.322,0.298


In [ ]:
# dictionary of country codes -> names w/ pycountry
country_mapping = {country.alpha_2: country.name for country in pycountry.countries}

# map abbreviations to full country names for multiple countries
def map_countries(abbrev_list):
    full_names = [country_mapping.get(abbrev.strip(), 'Unknown') for abbrev in abbrev_list.split(',')]
    return ', '.join(full_names)

# apply mapping function to 'availableCountries' col
streaming_df['availableCountriesFullName'] = streaming_df['availableCountries'].apply(map_countries)

streaming_df.head()

,title,type,genres,releaseYear,imdbId,imdbAverageRating,imdbNumVotes,availableCountries,service,availableCountriesFullName
0,The Fifth Element,movie,"Action, Adventure, Sci-Fi",1997.0,tt0119116,7.6,517631.0,"AT, CH, DE",Netflix,"Austria, Switzerland, Germany"
1,Kill Bill: Vol. 1,movie,"Action, Crime, Thriller",2003.0,tt0266697,8.2,1223205.0,"AE, AL, AO, AT, AU, AZ, BG, BH, BY, CA, CI, CM...",Netflix,"United Arab Emirates, Albania, Angola, Austria..."
2,Jarhead,movie,"Biography, Drama, War",2005.0,tt0418763,7.0,211839.0,"AD, AE, AG, AL, AO, AT, AZ, BA, BG, BH, BM, BR...",Netflix,"Andorra, United Arab Emirates, Antigua and Bar..."
3,Unforgiven,movie,"Drama, Western",1992.0,tt0105695,8.2,444402.0,"AU, BA, BG, CZ, HR, HU, MD, ME, MK, NZ, PL, RO...",Netflix,"Australia, Bosnia and Herzegovina, Bulgaria, C..."
4,Eternal Sunshine of the Spotless Mind,movie,"Drama, Romance, Sci-Fi",2004.0,tt0338013,8.3,1106220.0,"AD, AE, AG, AL, AO, AR, AU, AZ, BA, BB, BE, BG...",Netflix,"Andorra, United Arab Emirates, Antigua and Bar..."


In [ ]:
streaming_df['availableCountriesFullName'] = streaming_df['availableCountriesFullName'].str.split(", ")
exploded_df = streaming_df.explode('availableCountriesFullName')
titles_by_country_service = (
    exploded_df.groupby(['availableCountriesFullName', 'service'])
    .size()
    .reset_index(name='num_titles')
)

# merge w/ happiness indx
happiness_df['Country'] = happiness_df['Country or region']
merged_df = titles_by_country_service.merge(
    happiness_df[['Country', 'Score']],
    left_on='availableCountriesFullName',
    right_on='Country',
    how='left'
)

merged_df['Score'].fillna(0, inplace=True)

<ipython-input-115-f617b8480e0d>:18: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.





In [ ]:
merged_df.head()
# streaming_df['service'].value_counts()
# netflix: 20128
# hulu: 9845
# hbomax: 5753

,availableCountriesFullName,service,num_titles,Country,Score
0,Albania,Netflix,6944,Albania,4.719
1,Algeria,Netflix,6906,Algeria,5.211
2,Andorra,HBO Max,1597,NaN,0.000
3,Andorra,Netflix,6436,NaN,0.000
4,Angola,Netflix,6583,NaN,0.000


In [ ]:
def percentage(num_titles, service):
    if service == 'Netflix':
      return num_titles / 20128 * 100
    elif service == 'Hulu':
      return num_titles / 9845 * 100
    elif service == 'HBO Max':
      return num_titles / 5753 * 100
    else:
      return 0


merged_df['percentage'] = merged_df.apply(lambda x: percentage(x['num_titles'], x['service']),
                        axis=1)
merged_df.head()

,availableCountriesFullName,service,num_titles,Country,Score,percentage
0,Albania,Netflix,6944,Albania,4.719,34.499205
1,Algeria,Netflix,6906,Algeria,5.211,34.310413
2,Andorra,HBO Max,1597,NaN,0.000,27.759430
3,Andorra,Netflix,6436,NaN,0.000,31.975358
4,Angola,Netflix,6583,NaN,0.000,32.705684


In [ ]:
merged_df.columns

Index(['availableCountriesFullName', 'service', 'num_titles', 'Country',
       'Score', 'percentage'],
      dtype='object')

In [ ]:
merged_df.drop(columns=['Country'], inplace=True)


In [ ]:
merged_df.columns = [col.strip() for col in merged_df.columns]  # Removes whitespace
merged_df.rename(columns={'availableCountriesFullName': 'country'}, inplace=True)


In [ ]:
merged_df.head()

,country,service,num_titles,Score,percentage
0,Albania,Netflix,6944,4.719,34.499205
1,Algeria,Netflix,6906,5.211,34.310413
2,Andorra,HBO Max,1597,0.000,27.759430
3,Andorra,Netflix,6436,0.000,31.975358
4,Angola,Netflix,6583,0.000,32.705684


In [ ]:
fig = px.scatter(
    merged_df,
    x='num_titles',
    y='Score',
    hover_name='country',
    color='service',
    color_discrete_map={
        "HBO Max": "blue",
        "Netflix": "red",
        "Hulu": "green"
    },
    title="Scatter Plot of Titles vs Happiness Score by Streaming Service",
    hover_data={'Score': True},
    custom_data=['service']
)


fig.update_traces(
    hovertemplate="<b>%{hovertext}</b><br>" +
                  "Service: %{customdata[0]}<br>" +
                  "Number of Titles: %{x}<br>" +
                  "Happiness Score: %{y} / 10<extra></extra>"
)

fig.update_layout(
    xaxis_title="Number of Titles",
    yaxis_title="Happiness Score",
    legend_title="Streaming Service"
)

fig.show()


In [ ]:
fig = px.scatter(
    merged_df,
    x='percentage',
    y='Score',
    hover_name='country',
    color='service',
    color_discrete_map={
        "HBO Max": "blue",
        "Netflix": "red",
        "Hulu": "green"
    },
    title="Scatter Plot of Percentage of Total Titles Per Service vs Happiness Score in Each Country by Streaming Service",
    hover_data={'Score': True},
    custom_data=['service']
)

fig.update_traces(
    hovertemplate="<b>%{hovertext}</b><br>" +
                  "Service: %{customdata[0]}<br>" +
                  "Percentage: %{x}<br>" +
                  "Happiness Score: %{y} / 10<extra></extra>"
)

fig.update_layout(
    xaxis_title="Percentage",
    yaxis_title="Happiness Score",
    legend_title="Streaming Service"
)

fig.show()




In [ ]:
# latlon = pd.read_csv("/content/drive/MyDrive/latlon.csv")
latlon = pd.read_csv("latlon.csv")
latlon = latlon.drop(columns=['usa_state_code',	'usa_state_latitude',	'usa_state_longitude',	'usa_state', 'country_code'])


In [ ]:
merged_df = merged_df.merge(latlon, on="country", how="left")

In [ ]:
merged_df.head()

,country,service,num_titles,Score,percentage,latitude,longitude
0,Albania,Netflix,6944,4.719,34.499205,41.153332,20.168331
1,Algeria,Netflix,6906,5.211,34.310413,28.033886,1.659626
2,Andorra,HBO Max,1597,0.000,27.759430,42.546245,1.601554
3,Andorra,Netflix,6436,0.000,31.975358,42.546245,1.601554
4,Angola,Netflix,6583,0.000,32.705684,-11.202692,17.873887


In [ ]:
fig_map = px.scatter_geo(
    merged_df,
    lat='latitude',
    lon='longitude',
    hover_name='country',
    color='service',
    size='num_titles',
    projection="natural earth",
    color_discrete_map={
        "HBO Max": "blue",
        "Netflix": "red",
        "Hulu": "green"
    },
    hover_data={'Score': True, 'service': True}
)

fig_map.update_traces(
    hovertemplate="<b>%{hovertext}</b><br>" +
                  "Service: %{customdata[1]}<br>" +
                  "Number of Titles: %{marker.size}<br>" +
                  "Happiness Index: %{customdata[0]} / 10<extra></extra>"
)

fig_map.show()



In [ ]:
pivot_df = titles_by_country_service.pivot_table(
    index="availableCountriesFullName",
    columns="service",
    values="num_titles",
    fill_value=0
)

pivot_df.columns = [f"num_titles{col}" for col in pivot_df.columns]
pivot_df.reset_index(inplace=True)

In [ ]:
melted_df = pivot_df.melt(
    id_vars=["availableCountriesFullName"],
    value_vars=['num_titlesHBO Max', 'num_titlesNetflix', 'num_titlesHulu'],
    var_name="Service",
    value_name="Titles"
)

melted_df['Service'] = melted_df['Service'].replace({
    'num_titlesHBO Max': 'HBO Max',
    'num_titlesNetflix': 'Netflix',
    'num_titlesHulu': 'Hulu'
})

melted_df.rename(columns={"availableCountriesFullName": "Country"}, inplace=True)


barchart = px.bar(
    melted_df,
    x="Country",
    y="Titles",
    color="Service",
    barmode="group",
    title="Streaming Titles by Country",
    color_discrete_map={
        "HBO Max": "blue",
        "Netflix": "red",
        "Hulu": "green"
    }
)

barchart.update_layout(
    xaxis_title="Country",
    yaxis_title="Number of Titles",
    legend_title="Streaming Service"
)

barchart.show()


In [ ]:
country_to_plot = "Andorra"  # Change this to the desired country
filtered_df = pivot_df[pivot_df['availableCountriesFullName'] == country_to_plot]

melted_df = filtered_df.melt(
    id_vars=["availableCountriesFullName"],
    value_vars=['num_titlesHBO Max', 'num_titlesNetflix', 'num_titlesHulu'],
    var_name="Service",
    value_name="Titles"
)

melted_df['Service'] = melted_df['Service'].replace({
    'num_titlesHBO Max': 'HBO Max',
    'num_titlesNetflix': 'Netflix',
    'num_titlesHulu': 'Hulu'
})

barchart = px.bar(
    melted_df,
    x="Service",
    y="Titles",
    color="Service",
    title=f'Streaming Titles in {country_to_plot}',
    color_discrete_map={
        "HBO Max": "blue",
        "Netflix": "red",
        "Hulu": "green"
    }
)

barchart.update_layout(
    xaxis_title="Streaming Service",
    yaxis_title="Number of Titles",
    legend_title="Service"
)

barchart.show()



In [ ]:
# reload clean non-manipulated streaming data
# streaming_df2 = pd.read_csv("/content/drive/MyDrive/streaming.csv")
streaming_df2 = pd.read_csv("streaming.csv")
streaming_df2['availableCountriesFullName'] = streaming_df2['availableCountries'].apply(map_countries)

streaming_df2['availableCountriesFullName'] = streaming_df2['availableCountriesFullName'].fillna('Unknown').astype(str).str.split(", ")
exploded_df = streaming_df2.explode('availableCountriesFullName')

exploded_df['availableCountriesFullName'] = exploded_df['availableCountriesFullName'].str.strip()
exploded_df = exploded_df[exploded_df['availableCountriesFullName'] != 'Unknown']
exploded_df = exploded_df[exploded_df['availableCountriesFullName'] != 'nan']

# CHANGE COUNTRY NAME TO VARIABLE
selected_country = "United States"

country_df = exploded_df[exploded_df['availableCountriesFullName'] == selected_country]

country_df['genres'] = country_df['genres'].astype(str)
country_df = country_df[~country_df['genres'].str.contains(",")]

#calculate avg ratings
avg_ratings = (
    country_df.groupby(['genres', 'service'])['imdbAverageRating']
    .mean()
    .reset_index()
)

# top 10 genres
top_genres = (
    country_df['genres'].value_counts()
    .nlargest(10)
    .index
)
avg_ratings = avg_ratings[avg_ratings['genres'].isin(top_genres)]

# dataframe for heatmap
heatmap_data = avg_ratings.pivot(index='genres', columns='service', values='imdbAverageRating')
heatmap_data_clean = heatmap_data.dropna(how='all', axis=1)
heatmap_data_clean = heatmap_data_clean.dropna(how='all', axis=0)

#heatmap
fig = px.imshow(
    heatmap_data_clean.T,
    text_auto=".2f",
    color_continuous_scale="Greens",
    title="Average IMDb Rating by Genre and Streaming Service (United States)",
    labels={'x': 'Genres', 'y': 'Streaming Service', 'color': 'Avg IMDb Rating'}
)

# labs
fig.update_layout(
    font=dict(size=12),
    title_font=dict(size=16),
    xaxis=dict(
        tickangle=-45,
        automargin=True
    ),
    yaxis=dict(automargin=True),
    coloraxis_colorbar=dict(title="Avg IMDb Rating", ticks="outside")
)

fig.show()


<ipython-input-131-2fcfa2e1d14c>:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [ ]:
heatmap_data_clean.head()

service,HBO Max,Hulu,Netflix
genres,,,
Animation,7.520000,7.133333,6.558621
Comedy,6.822917,6.728402,6.216456
Crime,6.768421,7.145000,6.615385
Documentary,6.914187,6.948366,7.079605
Drama,7.031818,6.832292,6.565764


In [ ]:
external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css']
app = Dash(__name__, external_stylesheets=external_stylesheets)

styles = {
    'pre': {
        'border': 'thin lightgrey solid',
        'overflowX': 'scroll'
    }
}

pivot_df = titles_by_country_service.pivot_table(
    index="availableCountriesFullName",
    columns="service",
    values="num_titles",
    fill_value=0
)

pivot_df.columns = [f"num_titles{col}" for col in pivot_df.columns]
pivot_df.reset_index(inplace=True)

app = Dash(__name__)

map_fig = px.scatter_geo(
    merged_df,
    lat='latitude',
    lon='longitude',
    hover_name='country',
    color='service',
    size='num_titles',
    projection="natural earth",
    color_discrete_map={
        "HBO Max": "blue",
        "Netflix": "red",
        "Hulu": "green"
    },
    hover_data={'Score': True, 'service': True}
)

map_fig.update_traces(
    hovertemplate="<b>%{hovertext}</b><br>" +
                  "Service: %{customdata[1]}<br>" +
                  "Number of Titles: %{marker.size}<br>" +
                  "Happiness Index: %{customdata[0]} / 10<extra></extra>"
)


app.layout = html.Div([
    dcc.Graph(
        id='map-chart',
        figure=map_fig
    ),

    html.Div([
        html.Div([
            dcc.Graph(
                id='bar-chart'
            ),
        ], style={'width': '48%', 'display': 'inline-block', 'vertical-align': 'top'}),

        html.Div([
            dcc.Graph(
                id='heatmap-chart'
            ),
        ], style={'width': '48%', 'display': 'inline-block', 'vertical-align': 'top'}),
    ], style={'display': 'flex', 'justify-content': 'space-between', 'margin-top': '20px'}),
])


@app.callback(
    Output('bar-chart', 'figure'),
    Input('map-chart', 'clickData')
)
def update_bar_chart(clickData):
    if clickData is None:
        return px.bar(title="Click on a country to see details")

    country = clickData['points'][0]['hovertext']
    filtered_df = pivot_df[pivot_df['availableCountriesFullName'] == country]

    melted_df = filtered_df.melt(
        id_vars=["availableCountriesFullName"],
        value_vars=['num_titlesHBO Max', 'num_titlesNetflix', 'num_titlesHulu'],
        var_name="Service",
        value_name="Titles"
    )

    melted_df['Service'] = melted_df['Service'].replace({
        'num_titlesHBO Max': 'HBO Max',
        'num_titlesNetflix': 'Netflix',
        'num_titlesHulu': 'Hulu'
    })

    bar_fig = px.bar(
        melted_df,
        x="Service",
        y="Titles",
        color="Service",
        title=f"Streaming Titles in {country}",
        color_discrete_map={
            "HBO Max": "blue",
            "Netflix": "red",
            "Hulu": "green"
        }
    )
    bar_fig.update_layout(
        xaxis_title="Streaming Service",
        yaxis_title="Number of Titles",
        legend_title="Toggle Services"
    )

    return bar_fig



@app.callback(
    Output('heatmap-chart', 'figure'),
    Input('map-chart', 'clickData')
)
def update_heat_map(clickData):
    if clickData is None:
        return px.bar(title="Click on a country to see details")

    country = clickData['points'][0]['hovertext']

    country_df = exploded_df[exploded_df['availableCountriesFullName'] == country]

    country_df['genres'] = country_df['genres'].astype(str)
    country_df = country_df[~country_df['genres'].str.contains(",")]

    avg_ratings = (
        country_df.groupby(['genres', 'service'])['imdbAverageRating']
        .mean()
        .reset_index()
    )

    top_genres = (
        country_df['genres'].value_counts()
        .nlargest(10)
        .index
    )
    avg_ratings = avg_ratings[avg_ratings['genres'].isin(top_genres)]

    heatmap_data = avg_ratings.pivot(index='genres', columns='service', values='imdbAverageRating')
    heatmap_data_clean = heatmap_data.dropna(how='all', axis=1)
    heatmap_data_clean = heatmap_data_clean.dropna(how='all', axis=0)

    heatmap = px.imshow(
        heatmap_data_clean.T,
        text_auto=".2f",
        color_continuous_scale="Greens",
        title="Average IMDb Rating by Genre and Streaming Service ("+country+")",
        labels={'x': 'Genres', 'y': 'Streaming Service', 'color': 'Avg IMDb Rating'}
    )

    heatmap.update_layout(
        font=dict(size=12),
        title_font=dict(size=16),
        xaxis=dict(
            tickangle=-45,
            automargin=True
        ),
        yaxis=dict(automargin=True),
        coloraxis_colorbar=dict(title="Avg IMDb Rating", ticks="outside")
    )
    return heatmap

# if __name__ == '__main__':
#     app.run_server(debug=True)

!pip install pyngrok
from pyngrok import ngrok
!ngrok config add-authtoken 2po2a1ZpWT8nvn6F6NXHZmFdG4r_j7FntsuRFGgoXGMD6JcR

# Open an ngrok tunnel to port 8050
public_url = ngrok.connect(8050)
print("Dash app is live at:", public_url)

if __name__ == '__main__':
    app.run_server()

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
Dash app is live at: NgrokTunnel: "https://9e18-34-148-48-160.ngrok-free.app" -> "http://localhost:8050"


<IPython.core.display.Javascript object>

In [ ]:
app = Dash(__name__)

fig_scatter1 = px.scatter(
    merged_df,
    x='percentage',
    y='Score',
    hover_name='country',
    color='service',
    color_discrete_map={
        "HBO Max": "blue",
        "Netflix": "red",
        "Hulu": "green"
    },
    title="Scatter Plot of Percentage of Total Titles Per Service vs Happiness Score in Each Country by Streaming Service",
    hover_data={'Score': True},
    custom_data=['service']
)

fig_scatter1.update_traces(
    hovertemplate="<b>%{hovertext}</b><br>" +
                  "Service: %{customdata[0]}<br>" +
                  "Percentage: %{x}<br>" +
                  "Happiness Score: %{y} / 10<extra></extra>"
)

fig_scatter1.update_layout(
    xaxis_title="Percentage",
    yaxis_title="Happiness Score",
    legend_title="Streaming Service"
)

fig_scatter2 = px.scatter(
    merged_df,
    x='num_titles',
    y='Score',
    hover_name='country',
    color='service',
    color_discrete_map={
        "HBO Max": "blue",
        "Netflix": "red",
        "Hulu": "green"
    },
    title="Scatter Plot of Titles vs Happiness Score by Streaming Service",
    hover_data={'Score': True},
    custom_data=['service']
)

fig_scatter2.update_traces(
    hovertemplate="<b>%{hovertext}</b><br>" +
                  "Service: %{customdata[0]}<br>" +
                  "Number of Titles: %{x}<br>" +
                  "Happiness Score: %{y} / 10<extra></extra>"
)

fig_scatter2.update_layout(
    xaxis_title="Number of Titles",
    yaxis_title="Happiness Score",
    legend_title="Streaming Service"
)

melted_df = pivot_df.melt(
    id_vars=["availableCountriesFullName"],
    value_vars=['num_titlesHBO Max', 'num_titlesNetflix', 'num_titlesHulu'],
    var_name="Service",
    value_name="Titles"
)

melted_df['Service'] = melted_df['Service'].replace({
    'num_titlesHBO Max': 'HBO Max',
    'num_titlesNetflix': 'Netflix',
    'num_titlesHulu': 'Hulu'
})


melted_df.rename(columns={"availableCountriesFullName": "Country"}, inplace=True)


barchart = px.bar(
    melted_df,
    x="Country",
    y="Titles",
    color="Service",
    barmode="group",
    title="Streaming Titles by Country",
    color_discrete_map={
        "HBO Max": "blue",
        "Netflix": "red",
        "Hulu": "green"
    }
)

barchart.update_layout(
    xaxis_title="Country",
    yaxis_title="Number of Titles",
    legend_title="Streaming Service"
)

app.layout = html.Div([
    html.Div([
        html.Div([
            dcc.Graph(
                id='scatter1',
                figure=fig_scatter1
            )
        ], style={'width': '48%', 'display': 'inline-block', 'vertical-align': 'top'}),

        html.Div([
            dcc.Graph(
                id='scatter2',
                figure=fig_scatter2
            )
        ], style={'width': '48%', 'display': 'inline-block', 'vertical-align': 'top'}),
    ], style={'display': 'flex', 'justify-content': 'space-between', 'margin-bottom': '20px'}),

    html.Div([
        dcc.Graph(
            id='barchart',
            figure=barchart
        )
    ], style={'width': '100%', 'display': 'block'})
])

if __name__ == '__main__':
    app.run_server(debug=True)

# !pip install pyngrok
# from pyngrok import ngrok
# !ngrok config add-authtoken 2po2a1ZpWT8nvn6F6NXHZmFdG4r_j7FntsuRFGgoXGMD6JcR

# # Open an ngrok tunnel to port 8050
# public_url = ngrok.connect(8050)
# print("Dash app is live at:", public_url)

# if __name__ == '__main__':
#     app.run_server()



<IPython.core.display.Javascript object>